# WET-004: Donut Deblending Test

Owner: **Bryce Kalmbach** <br>
Last Verified to Run: **2025-03-27** <br>
Software Version:
  - `ts_wep`: **14.1.1** 
  - `lsst_distrib`: **w_2025_11**

In [ ]:
# Times Square Parameters
collection_name = 'u/brycek/aosRefitWcs_danish_singleBlends_80pxMinSep'
detector = 0
min_seq_num = 90
max_seq_num = 107
day_obs = 20241115

In [ ]:
import numpy as np
from copy import copy
from astropy.visualization import ZScaleInterval
from matplotlib import pyplot as plt
from lsst.daf.butler import Butler

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.palettes import Viridis256
from bokeh.models import LinearColorMapper
output_notebook()
%matplotlib inline

In [ ]:
butler = Butler('/repo/main')

In [ ]:
camera = butler.get('camera', {'instrument': "LSSTComCam"}, collections=collection_name)

## Identify Blended Donuts

In [ ]:
refs_with_visit_tables = butler.query_datasets('refitWcsDonutTable', 
                                             collections=collection_name,
                                             where=f"exposure.day_obs = {day_obs} and exposure.seq_num >= {min_seq_num} and exposure.seq_num <= {max_seq_num} and instrument = 'LSSTComCam'")

In [ ]:
len(refs_with_visit_tables)

In [ ]:
ref_on = 0
dt = butler.get(refs_with_visit_tables[ref_on])

In [ ]:
blended_idx = list()
for idx, loc in enumerate(dt.meta['blend_centroid_x']):
    if len(loc) > 0:
        blended_idx.append(idx)

In [ ]:
blended_idx

In [ ]:
dataId = refs_with_visit_tables[ref_on].dataId.to_simple().dataId
dataId['exposure'] = dataId['visit']
dataId['detector']

In [ ]:
post_isr = butler.get('postISRCCD', dataId=dataId, collections=collection_name)

In [ ]:
# fig = plt.figure(figsize=(12,12))
# plt.imshow(post_isr.image.array, vmax=4500)
# plt.scatter(dt['centroid_x'][blended_idx], dt['centroid_y'][blended_idx], s=120, c='r')

In [ ]:
p = figure(width=1000, height=1000)
p.x_range.range_padding = p.y_range.range_padding = 0
color_mapper = LinearColorMapper(palette=Viridis256, low=2000, high=10000)
p.image(image=[post_isr.image.array], x=0, y=0, dw=10, dh=10, level="image", color_mapper=color_mapper)
show(p)